# **Modelos de Aprendizaje Supervisado**

In [ ]:
import os
import kagglehub
import numpy as np
import pandas as pd

# 1. Descarga del dataset
path = kagglehub.dataset_download("fedesoriano/stroke-prediction-dataset")
files = os.listdir(path)
csv_files = [f for f in files if f.endswith('.csv')]
csv_path = os.path.join(path, csv_files[0])
df = pd.read_csv(csv_path)

# 2. Imputación de bmi
bmi_mean = df['bmi'].mean()
df['bmi'] = df['bmi'].fillna(bmi_mean)

# 3. Codificación One-Hot y eliminación de id
df = pd.get_dummies(df)
if 'id' in df.columns:
    df.drop(columns=['id'], inplace=True)

print("Dataset preparado correctamente. Forma:", df.shape)

Using Colab cache for faster access to the 'stroke-prediction-dataset' dataset.
Dataset preparado correctamente. Forma: (5110, 22)


**Sección 1:** Carga e Inspección Inicial con pandas

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Separar variables predictoras (X) y variable objetivo (y)
X = df.drop(columns=['stroke'])
y = df['stroke']

# 2. División train/test (80/20) manteniendo la proporción de clases
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# 3. Escalado de variables
scaler = StandardScaler()

# Se ajusta solo con los datos de entrenamiento para evitar fuga de información (data leakage)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Tamaño X_train: {X_train_scaled.shape} | Tamaño X_test: {X_test_scaled.shape}")

Tamaño X_train: (4088, 21) | Tamaño X_test: (1022, 21)


**Sección 2:** Diseño de la Arquitectura y Selección de Funciones (Keras / TensorFlow)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Construcción de la arquitectura secuencial
model = keras.Sequential([
    layers.Input(shape=(X_train_scaled.shape[1],)),

    # Primera capa oculta
    layers.Dense(units=32, activation='relu'),

    # Segunda capa oculta
    layers.Dense(units=16, activation='relu'),

    # Capa de salida para clasificación binaria
    layers.Dense(units=1, activation='sigmoid')
])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 32)             │           704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,249 (4.88 KB)

 Trainable params: 1,249 (4.88 KB)

 Non-trainable params: 0 (0.00 B)

**Sección 3:** Entrenamiento, Métricas y Optimizadores


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

# Compilación del modelo orientada a métricas para desbalance
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[
        keras.metrics.AUC(name='auc'),
        keras.metrics.Recall(name='recall'),
        'accuracy'
    ]
)

# Callback para detener el entrenamiento cuando no haya mejora
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

# Entrenamiento del modelo
history = model.fit(
    X_train_scaled,
    y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/100
103/103 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9309 - auc: 0.5026 - loss: 0.2912 - recall: 0.0066 - val_accuracy: 0.9425 - val_auc: 0.5657 - val_loss: 0.2226 - val_recall: 0.0000e+00
Epoch 2/100
103/103 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9535 - auc: 0.7137 - loss: 0.1742 - recall: 0.0000e+00 - val_accuracy: 0.9425 - val_auc: 0.7237 - val_loss: 0.2017 - val_recall: 0.0000e+00
Epoch 3/100
103/103 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9535 - auc: 0.7827 - loss: 0.1636 - recall: 0.0000e+00 - val_accuracy: 0.9425 - val_auc: 0.7487 - val_loss: 0.1987 - val_recall: 0.0000e+00
Epoch 4/100
103/103 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9535 - auc: 0.8096 - loss: 0.1587 - recall: 0.0000e+00 - val_accuracy: 0.9425 - val_auc: 0.7705 - val_loss: 0.1954 - val_recall: 0.0000e+00
Epoch 5/100
103/103 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9535 - auc: 0.8262 - loss: 0.1545 - recall: 0.0000e+00 - val_accuracy: 0.9425 - val_auc: 0.7783 - val_loss: 0